# Why the "grid" layout scores so high — and why it's a `recon` artifact

This notebook reproduces the diagnostics behind a puzzle: the deterministic **grid** detector
layout scores **U ≈ 136** under the trained surrogate + reconstruction net, while every other
layout (center, rings, perturbed, even the *same grid relabeled*) scores ~25–70 — and the
optimizers (Adam + L-BFGS, differential evolution) never reach 136.

**Punchline (proven below):** the high score is **not physical**. The reconstruction network
(`Reconstruction`, a flat MLP) is *not* permutation-invariant despite permutation augmentation
during training, so the grid's 136 is an artifact of evaluating it in **one specific detector
ordering**. The same physical grid, relabeled, scores ~30 — exactly where the optimizers land.

Run with the project conda env kernel: `/n/home05/zdimitrov/.conda/envs/multiproc_env/bin/python`.
All cells are CPU-only and cheap (batch ≤ 8000 showers).

## Setup
Load the optimization module by path (its filename starts with a digit, so it can't be `import`ed
normally). This also pulls in the layout strategies used to build the training set.

In [ ]:
import os, sys, glob, json, math
import numpy as np, torch
torch.set_num_threads(2)

_HERE = os.path.abspath("../")
if _HERE not in sys.path:
    sys.path.insert(0, _HERE)
    
V6 = "/n/home05/zdimitrov/tambo/TambOpt/detector_optimization_v6"
if V6 not in sys.path:
    sys.path.insert(0, V6)

import importlib.util
spec = importlib.util.spec_from_file_location("de", os.path.join(V6, "04_optimize_differential_evolution_pop.py"))
de = importlib.util.module_from_spec(spec); spec.loader.exec_module(de)
import modules_v6.detector_strategies_ne as DS

print("loaded module; DEVICE =", de.DEVICE, " N_DETECTORS =", de.N_DETECTORS)

loaded module; DEVICE = cpu  N_DETECTORS = 100


## Load the frozen models under test + mountain + primaries
These are exactly the artifacts the optimizers see: the dual-species DeepSets surrogate
(`fnn_electron.pt` + `fnn_muon.pt`), the reconstruction MLP (`recon.pt`), and the mountain geometry.
`U_layout(N, E)` is the composite utility that the layout optimizers maximize.

In [ ]:
fnn, recon = de.load_models(de.DEVICE)
mtn = de.load_tr_mountain(de.GEOMETRY_PATH_RESOLVED, de.GEOMETRY_GROUP, de.DET_KEY,
    east_entry=de.EAST_ENTRY, layer_east_dx=de.LAYER_EAST_DX, n_planes=de.N_PLANES)
prim = torch.load(os.path.join(de.TRAINING_DATASET_FOLDER, "primary.pt")).float()
Ntot = prim.shape[0]

# One fixed 2000-shower batch. U is batch-size independent (see Test 1), so 2000 is plenty.
pb = prim[torch.randperm(Ntot, generator=torch.Generator().manual_seed(42))[:2000]]

def project(N, E):
    return de.project_to_mountain_ne(mtn, torch.as_tensor(N).float(), torch.as_tensor(E).float())

def U_layout(N, E, batch=None, do_project=True):
    """Composite utility U of a (North, East) layout against a primary batch."""
    if do_project:
        N, E = project(N, E)
    u, r, _ = de.utility_of_xy(torch.as_tensor(N).float(), torch.as_tensor(E).float(),
                               pb if batch is None else batch, fnn, recon)
    return float(u)

def clean(scheme):
    N, E = de.sample_initial_layout_ne(mtn, n_units=de.N_DETECTORS, scheme=scheme)
    return project(N, E)

print("models + mountain loaded; primaries:", Ntot)
print("mountain East bbox [%.0f, %.0f]  Up range [%.0f, %.0f]" % (
    mtn.east_lo, mtn.east_hi, mtn.centroids_NUE[:,1].min(), mtn.centroids_NUE[:,1].max()))

## Test 1 — U is a per-event mean, so it does NOT depend on batch size
`U_E` / `U_angle` average over events, so the objective's *level* is independent of how many
showers you score. This rules out the 50k-vs-512 fixed-batch difference (between
`04_optimize_differential_evolution_pop.py` and the others) as a cause of any U-level difference —
only the **layout** matters.

In [3]:
gN, gE = clean("grid")
for B in (256, 512, 2000, 8000):
    b = prim[torch.randperm(Ntot, generator=torch.Generator().manual_seed(42))[:B]]
    print(f"B={B:6d}  U={U_layout(gN, gE, batch=b, do_project=False):+.2f}")

B=   256  U=+140.50


B=   512  U=+136.95


B=  2000  U=+135.67


B=  8000  U=+137.09


## Test 2 — the clean grid scores ~136; center / perturbed score ~25–34
The puzzle in one cell: among hand-built layouts the grid is dramatically higher than a center
cluster or a 1000 m-perturbed grid. (`r_mean` saturates at 1 because `RECONSTRUCT_THRESHOLD=10`
with 100 detectors, so U is driven by angular/energy reconstruction accuracy, not detector count.)

In [4]:
gN, gE = clean("grid"); cN, cE = clean("center")
gen = torch.Generator().manual_seed(1)
pN, pE = project(gN + torch.randn(100, generator=gen)*1000, gE + torch.randn(100, generator=gen)*1000)
for nm, (N, E) in [("clean grid", (gN, gE)), ("clean center", (cN, cE)),
                   ("perturbed grid s=1000", (pN, pE))]:
    print(f"{nm:22s} U={U_layout(N, E, do_project=False):+.2f}")

clean grid             U=+135.67


clean center           U=+24.98


perturbed grid s=1000  U=+33.63


## Test 3 — peak shape: how fast U falls as the grid is jittered
Add i.i.d. Gaussian noise of std σ (metres) to the grid. U decays smoothly but steeply — a ~200 m
jitter (≈4% of the array's ~4600 m span) already roughly halves it. A genuine physics optimum
would not be this fragile; this sharpness is the first hint of a surrogate artifact.

In [5]:
gN, gE = clean("grid")
for s in (0, 25, 50, 100, 200, 400, 800):
    if s == 0:
        print(f"sigma={s:4d}  U={U_layout(gN, gE, do_project=False):+.2f}"); continue
    us = []
    for seed in range(3):
        g = torch.Generator().manual_seed(seed)
        us.append(U_layout(gN + torch.randn(100, generator=g)*s, gE + torch.randn(100, generator=g)*s))
    print(f"sigma={s:4d}  U={np.mean(us):+.2f} (+/-{np.std(us):.1f})")

sigma=   0  U=+135.67


sigma=  25  U=+127.08 (+/-1.3)


sigma=  50  U=+113.63 (+/-4.3)


sigma= 100  U=+95.64 (+/-4.2)


sigma= 200  U=+76.70 (+/-4.7)


sigma= 400  U=+54.36 (+/-3.0)


sigma= 800  U=+33.12 (+/-4.9)


## Test 4 — U of the 7 ACTUAL training layout strategies
The surrogate/recon were trained on these 7 layout families (grid ± jitter, center clusters, and
**three ring radii** up to 1800 m — i.e. the training set DOES include spread-out layouts). Yet
only the tight grid (`grid_jit20`) is high; the spread rings and center clusters all sit at
~56–75. So "spread coverage" by itself does not explain the grid's score.

In [6]:
for name, fn_name, kw in DS._STRATEGIES:
    fn = DS._STRATEGY_FNS[fn_name]
    us = [U_layout(*fn(mtn, n_det=de.N_DETECTORS, rng=np.random.default_rng(s), **kw), do_project=False)
          for s in range(4)]
    print(f"{name:16s} U={np.mean(us):+7.2f} (+/-{np.std(us):4.1f})   {kw}")
print(f"\nclean grid (no jitter): U={U_layout(*clean('grid'), do_project=False):+.2f}")

grid_jit20       U=+131.24 (+/- 0.5)   {'jitter_sigma': 20.0}


grid_jit200      U= +75.30 (+/- 3.6)   {'jitter_sigma': 200.0}


center_gauss200  U= +60.62 (+/- 1.2)   {'sigma': 200.0}


center_gauss400  U= +56.07 (+/- 4.3)   {'sigma': 400.0}


rings_R300       U= +61.86 (+/- 3.7)   {'outer_radius': 300.0, 'n_rings': 5, 'jitter_sigma': 200.0}


rings_R800       U= +68.79 (+/- 3.2)   {'outer_radius': 800.0, 'n_rings': 6, 'jitter_sigma': 200.0}


rings_R1800      U= +63.83 (+/- 2.8)   {'outer_radius': 1800.0, 'n_rings': 8, 'jitter_sigma': 200.0}



clean grid (no jitter): U=+135.67


## Test 5 — what the optimizers actually converge to (saved run logs)
The DE and L-BFGS pipelines plateau at ~55–70 — the level of the *generic* training layouts
(rings / center / jittered grid), not the grid's 136. The exception, `de_population` ≈ 139, only
"wins" because the clean grid is placed directly in its initial population (it doesn't optimize to
it).

In [7]:
RD = "/n/holylfs05/LABS/arguelles_delgado_lab/Everyone/zdimitrov/detector_optimization_v6/optimization_runs_200k_dual"
for lj in sorted(glob.glob(os.path.join(RD, "*", "optimize_log.json"))):
    J = json.load(open(lj))
    nm = os.path.basename(os.path.dirname(lj)).replace("test_v6_run_04_optimize_", "")
    bu = J.get("best_U")
    print(f"{nm:24s} best_U={bu:+.2f}" if isinstance(bu,(int,float)) else f"{nm:24s} best_U={bu}")

de_ensemble_center       best_U=+54.91
de_ensemble_grid         best_U=+56.21
de_population            best_U=+138.87


lbfgs_ensemble_center    best_U=+66.30


lbfgs_ensemble_combined  best_U=+69.97


lbfgs_ensemble_grid      best_U=+69.97


## Test 6 — THE SMOKING GUN: permute the detector ORDER, keep positions identical
Keep the exact same 100 grid positions; only reorder which detector is fed into which slot of
`recon`. A physically correct (permutation-invariant) reconstructor MUST return the same U —
relabeling detectors can't change the physics. It doesn't: **136 → ~32**. The score depends on
detector labeling, which is unphysical.

In [8]:
gN, gE = clean("grid")
print(f"grid, CANONICAL order:  U={U_layout(gN, gE, do_project=False):+.2f}")
for s in range(3):
    p = torch.randperm(100, generator=torch.Generator().manual_seed(s))
    print(f"  permuted order seed{s}: U={U_layout(gN[p], gE[p], do_project=False):+.2f}")

grid, CANONICAL order:  U=+135.67


  permuted order seed0: U=+32.40


  permuted order seed1: U=+33.38


  permuted order seed2: U=+32.70


## Test 7 — it's NOT coverage: rigid shift + naive lattice
A rigid shift of the whole grid (regularity and coverage preserved) decays U smoothly; and a naive
10×10 lattice with good coverage (91/100 distinct positions after mountain projection) still scores
~22. So poor coverage / clumping is not why the other layouts are low.

In [9]:
gN, gE = clean("grid")
for d in (100, 200, 500, 1000):
    print(f"rigid shift +{d:4d} m North: U={U_layout(gN + d, gE):+.2f}")

ax = torch.linspace(float(gN.min()), float(gN.max()), 10)
ay = torch.linspace(float(gE.min()), float(gE.max()), 10)
GX, GY = torch.meshgrid(ax, ay, indexing="ij")
LN, LE = GX.reshape(-1), GY.reshape(-1)
LNp, LEp = project(LN, LE)
uniq = torch.unique(torch.stack([LNp, LEp], -1).round(), dim=0).shape[0]
print(f"naive 10x10 lattice: U={U_layout(LNp, LEp, do_project=False):+.2f}   "
      f"unique positions after projection = {uniq}/100")

rigid shift + 100 m North: U=+112.89


rigid shift + 200 m North: U=+87.73


rigid shift + 500 m North: U=+45.49


rigid shift +1000 m North: U=+27.95


naive 10x10 lattice: U=+21.61   unique positions after projection = 91/100


## Test 8 — measure `recon`'s permutation-(non)invariance directly
Training **does** apply per-batch detector permutation augmentation (`03_train_recon.py:425`,
intended to make the flat MLP invariant), and the loaded `recon.pt` was trained with it (it is not
a stale checkpoint). Yet permuting the detector rows changes recon's raw output by **33–68%** of its
magnitude — so the augmentation did NOT produce an invariant function. This is the documented
flat-MLP failure mode (a plain MLP can't be made truly permutation-invariant by augmentation
alone), and it is the root cause of the whole puzzle.

In [10]:
def feats(N, E):
    N, E = project(N, E)
    xy = torch.stack([N, E], -1).unsqueeze(0).expand(pb.shape[0], -1, -1)
    pr = fnn(pb, xy)
    return torch.stack([xy[..., 0], xy[..., 1], pr[..., 0], pr[..., 1]], -1)   # (B, 100, 4)

def invariance(name, N, E):
    f = feats(torch.as_tensor(N).float(), torch.as_tensor(E).float())
    o1 = recon(f.reshape(f.shape[0], -1))
    ds = []
    for s in range(5):
        p = torch.randperm(100, generator=torch.Generator().manual_seed(s))
        o2 = recon(f[:, p, :].reshape(f.shape[0], -1))
        ds.append((o1 - o2).abs().mean().item())
    print(f"{name:12s} mean|delta out|={np.mean(ds):.4f}  (|out|~{o1.abs().mean().item():.3f})  "
          f"rel={np.mean(ds)/o1.abs().mean().item():.3f}")

invariance("grid",       *de.sample_initial_layout_ne(mtn, 100, "grid"))
invariance("rings_R800", *DS.layout_rings(mtn, n_det=100, rng=np.random.default_rng(0),
                                          outer_radius=800.0, n_rings=6, jitter_sigma=200.0))
invariance("center",     *DS.layout_center_gaussian(mtn, n_det=100, rng=np.random.default_rng(0),
                                                     sigma=200.0))

grid         mean|delta out|=0.3332  (|out|~0.485)  rel=0.686


rings_R800   mean|delta out|=0.1567  (|out|~0.459)  rel=0.342


center       mean|delta out|=0.1489  (|out|~0.443)  rel=0.336


## Test 9 — the fair (order-averaged) value of the grid
Physics is order-invariant, so the honest score of a layout is its average over detector orderings.
The grid's canonical-order 136 is a **~36σ outlier** above its own order-average (~30). The grid is
really worth ~30 under this recon — exactly the level the optimizers reach. **There is nothing for
them to "find".**

In [11]:
gN, gE = clean("grid")
canonical = U_layout(gN, gE, do_project=False)
print(f"grid, CANONICAL order:  U={canonical:+.2f}")
us = []
for s in range(15):
    p = torch.randperm(100, generator=torch.Generator().manual_seed(s))
    us.append(U_layout(gN[p], gE[p], do_project=False))
us = np.array(us)
print(f"grid, 15 RANDOM orders: mean={us.mean():+.2f}  std={us.std():.2f}  "
      f"min={us.min():+.2f}  max={us.max():+.2f}")
print(f"-> canonical is {(canonical - us.mean())/us.std():.0f} sigma above the order-average")

grid, CANONICAL order:  U=+135.67


grid, 15 RANDOM orders: mean=+30.08  std=2.93  min=+22.81  max=+33.91
-> canonical is 36 sigma above the order-average


## Conclusions

1. **The grid is not actually better.** Its U≈136 is an artifact of evaluating it in one specific
   detector ordering. The *same physical grid* relabeled scores ~30 (Test 6, Test 9) — a ~36σ drop.
2. **Root cause:** `recon` (`Reconstruction`, a flat MLP) is **not permutation-invariant**, even
   though `03_train_recon.py` applies permutation augmentation. Measured residual order-dependence
   is 33–68% of the output magnitude (Test 8).
3. **It's not batch size** (Test 1) and **not coverage/clumping** (Test 7).
4. **The optimizers are behaving correctly:** they reach ~55–70 (Test 5) = the recon's real,
   order-averaged quality level (~30–70 depending on layout). The grid's extra points are a
   non-physical spike at one input ordering that no position-space optimizer can (or should) chase.

**Fix:** replace the flat-MLP `recon` with a permutation-invariant architecture (DeepSets /
attention over detectors — the same fix already applied to the surrogate), so the utility becomes a
trustworthy, order-invariant function of the layout. Until then, layout optimization on this `recon`
is optimizing a labeling artifact, not detector performance.

## Test 10 -What does the permutation function do?

In [ ]:
import importlib.util
spec_03 = importlib.util.spec_from_file_location("recon_imports", os.path.join(V6, "03_train_recon.py"))
recon_imports = importlib.util.module_from_spec(spec_03); spec_03.loader.exec_module(recon_imports)

In [ ]:
def permute_detectors_recon(xy: torch.Tensor,
                            E:  torch.Tensor,
                            T:  torch.Tensor
                            ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Independent per-sample random permutation of the 100 detectors.

    The recon target is a scalar 4-vector (dir_x, dir_y, dir_z, log_e_norm) —
    invariant under detector permutations — so we do NOT
    permute the target. The network learns to produce the same output for any
    ordering of its per-detector features.
    """
    B, n_det, _ = xy.shape
    rand_key = torch.rand(B, n_det, device=xy.device)
    perms = torch.argsort(rand_key, dim=1)
    idx_xy = perms.unsqueeze(-1).expand(-1, -1, 2)
    xy_p = torch.gather(xy, 1, idx_xy)
    E_p  = torch.gather(E,  1, perms)
    T_p  = torch.gather(T,  1, perms)
    return xy_p, E_p, T_p

In [4]:
gN, gE = clean("grid")
xy = torch.stack([gN, gE], -1).unsqueeze(0).expand(pb.shape[0], -1, -1)

In [ ]:
B, n_det, _ = xy.shape
rand_key = torch.rand(B, n_det, device=xy.device)
perms = torch.argsort(rand_key, dim=1)
idx_xy = perms.unsqueeze(-1).expand(-1, -1, 2)
xy_p = torch.gather(xy, 1, idx_xy)

In [13]:
torch.argsort(rand_key, dim=1)[0]

tensor([41, 74, 50, 35, 82, 67, 94, 51, 76, 80, 16, 14, 83, 20, 26,  3, 22, 66,
        28, 55, 60, 81, 93, 19, 44, 30, 40, 52, 34, 12, 29, 21, 23,  7, 48, 97,
        27, 57, 11, 92, 89, 36,  1,  0, 49, 58, 99, 59, 32, 68, 88, 37,  4, 70,
        15, 73, 45, 69, 86, 61, 78, 39,  5, 38,  2, 84, 75, 71, 53, 77, 91, 46,
        95, 72, 79, 65,  9, 47, 90, 43, 96, 25,  6, 13, 54, 33, 98, 31, 85, 63,
        64, 87, 62, 17, 10, 42, 56,  8, 18, 24])

In [22]:
xy.shape

torch.Size([2000, 100, 2])

In [25]:
xy[0][41]

tensor([-393.9576, -324.2709])

In [23]:
xy_p[0]

tensor([[ -393.9576,  -324.2709],
        [  753.4175,   428.8102],
        [  370.9591,  -136.0007],
        [  562.1883,  -512.5412],
        [  753.4175,   617.0804],
        [ 1135.8759,   240.5399],
        [-1158.8743,   993.6210],
        [  944.6467,  -136.0007],
        [ 1900.7925,   428.8102],
        [ -393.9576,   617.0804],
        [ 1518.3341, -1077.3521],
        [  370.9591, -1077.3521],
        [ 1518.3341,   617.0804],
        [  179.7299,  -889.0818],
        [ -202.7284,  -700.8115],
        [ 1327.1050, -1642.1630],
        [ 1518.3341,  -889.0818],
        [  562.1883,   240.5399],
        [  944.6467,  -700.8115],
        [-1350.1034,    52.2696],
        [ 1709.5634,    52.2696],
        [  179.7299,   617.0804],
        [-1732.5618,   993.6210],
        [ -393.9576,  -889.0818],
        [ 1327.1050,  -324.2709],
        [ 2092.0217,  -700.8115],
        [-1158.8743,  -324.2709],
        [ 1518.3341,  -136.0007],
        [  -11.4992,  -512.5412],
        [ -776

In [21]:
perms.unsqueeze(-1).expand(-1, -1, 2)[0]

tensor([[41, 41],
        [74, 74],
        [50, 50],
        [35, 35],
        [82, 82],
        [67, 67],
        [94, 94],
        [51, 51],
        [76, 76],
        [80, 80],
        [16, 16],
        [14, 14],
        [83, 83],
        [20, 20],
        [26, 26],
        [ 3,  3],
        [22, 22],
        [66, 66],
        [28, 28],
        [55, 55],
        [60, 60],
        [81, 81],
        [93, 93],
        [19, 19],
        [44, 44],
        [30, 30],
        [40, 40],
        [52, 52],
        [34, 34],
        [12, 12],
        [29, 29],
        [21, 21],
        [23, 23],
        [ 7,  7],
        [48, 48],
        [97, 97],
        [27, 27],
        [57, 57],
        [11, 11],
        [92, 92],
        [89, 89],
        [36, 36],
        [ 1,  1],
        [ 0,  0],
        [49, 49],
        [58, 58],
        [99, 99],
        [59, 59],
        [32, 32],
        [68, 68],
        [88, 88],
        [37, 37],
        [ 4,  4],
        [70, 70],
        [15, 15],
        [7